<a href="https://colab.research.google.com/github/arnavnigam31/DeepLearningLab/blob/main/expt10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
with open("tshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

block_size = 128
batch_size = 32

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]

        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(query)

        values = values.view(N, value_len, self.heads, self.head_dim)
        keys = keys.view(N, key_len, self.heads, self.head_dim)
        queries = queries.view(N, query_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", queries, keys)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=-1)

        out = torch.einsum("nhql,nlhd->nqhd", attention, values)
        out = out.reshape(N, query_len, self.heads * self.head_dim)

        return self.fc_out(out)

In [4]:
class FeedForward(nn.Module):
    def __init__(self, embed_size, forward_expansion):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, embed_size, 2) *
                             -(math.log(10000.0) / embed_size))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = FeedForward(embed_size, forward_expansion)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn))

        forward = self.ff(x)
        out = self.norm2(x + self.dropout(forward))
        return out

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, heads, forward_expansion, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)

        self.layers = nn.ModuleList([
            EncoderBlock(embed_size, heads, forward_expansion, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = self.dropout(self.pos(self.embedding(x)))

        for layer in self.layers:
            out = layer(out, mask)

        return out

In [8]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout):
        super().__init__()

        self.self_attn = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)

        self.enc_attn = MultiHeadAttention(embed_size, heads)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = FeedForward(embed_size, forward_expansion)
        self.norm3 = nn.LayerNorm(embed_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        attn = self.self_attn(x, x, x, trg_mask)
        x = self.norm1(x + self.dropout(attn))

        attn2 = self.enc_attn(enc_out, enc_out, x, src_mask)
        x = self.norm2(x + self.dropout(attn2))

        forward = self.ff(x)
        out = self.norm3(x + self.dropout(forward))
        return out

In [9]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, heads, forward_expansion, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)

        self.layers = nn.ModuleList([
            DecoderBlock(embed_size, heads, forward_expansion, dropout)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        x = self.dropout(self.pos(self.embedding(x)))

        for layer in self.layers:
            x = layer(x, enc_out, src_mask, trg_mask)

        return self.fc_out(x)

In [10]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_size=256, num_layers=4, heads=8,
                 forward_expansion=4, dropout=0.1):
        super().__init__()

        self.encoder = Encoder(vocab_size, embed_size, num_layers, heads, forward_expansion, dropout)
        self.decoder = Decoder(vocab_size, embed_size, num_layers, heads, forward_expansion, dropout)

    def make_trg_mask(self, trg):
        N, trg_len = trg.shape
        mask = torch.tril(torch.ones((trg_len, trg_len))).expand(N, 1, trg_len, trg_len)
        return mask

    def forward(self, src, trg):
        src_mask = None
        trg_mask = self.make_trg_mask(trg)

        enc_out = self.encoder(src, src_mask)
        out = self.decoder(trg, enc_out, src_mask, trg_mask)
        return out

In [11]:

model = Transformer(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for step in range(2000):
    xb, yb = get_batch("train")
    xb, yb = xb, yb

    logits = model(xb, xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print("Step:", step, "Loss:", loss.item())

Step: 0 Loss: 4.725736618041992
Step: 100 Loss: 2.392390012741089
Step: 200 Loss: 2.2315571308135986
Step: 300 Loss: 1.890639305114746
Step: 400 Loss: 1.2272355556488037
Step: 500 Loss: 0.9327107667922974
Step: 600 Loss: 0.7000187635421753
Step: 700 Loss: 0.5599520206451416
Step: 800 Loss: 0.4624464213848114
Step: 900 Loss: 0.3646946847438812
Step: 1000 Loss: 0.3095483183860779
Step: 1100 Loss: 0.27922889590263367
Step: 1200 Loss: 0.23431873321533203
Step: 1300 Loss: 0.21872323751449585
Step: 1400 Loss: 0.21716605126857758
Step: 1500 Loss: 0.2379491925239563
Step: 1600 Loss: 0.14062799513339996
Step: 1700 Loss: 0.17033815383911133
Step: 1800 Loss: 0.1286856234073639
Step: 1900 Loss: 0.15285810828208923


In [13]:
def generate(model, start="To be", max_new_tokens=200):
    model.eval()
    idx = torch.tensor([encode(start)], dtype=torch.long)

    for _ in range(max_new_tokens):
        logits = model(idx, idx)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return decode(idx[0].tolist())

print(generate(model))

To be oe oe oe oee oeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTeeeeeeeeeeeeeeeee eeeeee eeeeeeeeeeeeeeeeeeeeeteeeeeeeeeee eeeeeeeeeeeeeeeeeeeee
